In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset
import os

In [2]:
# Get the directory of the current notebook (if it's in the same folder as the CSVs)
project_dir = os.getcwd()
# Now construct the path reliably
normal_csv_path = os.path.join(project_dir, '../data/normal2.csv')
slowed_csv_path = os.path.join(project_dir, '../data/slowed2.csv')

In [3]:
def prepare_data(file_path, label_val, window_size=128):
    df = pd.read_csv(file_path, header=None, names=['raw', 'baseline', 'delta', 'current'])
    data = df['current'].values.reshape(-1, 1)
    scaler = MinMaxScaler()
    data = scaler.fit_transform(data)
    
    windows, labels = [], []
    for i in range(len(data) - window_size):
        windows.append(data[i:i+window_size].T) # Shape [1, 128]
        labels.append(label_val)
    return torch.tensor(np.array(windows), dtype=torch.float32), torch.tensor(labels, dtype=torch.long)

In [4]:
# Load data
X_n, y_n = prepare_data(normal_csv_path, 0)
X_s, y_s = prepare_data(slowed_csv_path, 1)
X = torch.cat([X_n, X_s], dim=0)
y = torch.cat([y_n, y_s], dim=0)

loader = DataLoader(TensorDataset(X, y), batch_size=32, shuffle=True)

In [5]:
class MotorFaultCNN(nn.Module):
    def __init__(self):
        super(MotorFaultCNN, self).__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=5)
        self.bn1 = nn.BatchNorm1d(32) # Added
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(2)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3)
        self.bn2 = nn.BatchNorm1d(64) # Added
        self.fc = nn.Linear(64 * 30, 2)
        
    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)
        return self.fc(x)

In [6]:
model = MotorFaultCNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [7]:
print("Starting training...")
for epoch in range(10): # 10 epochs
    total_loss = 0
    for batch_X, batch_y in loader:
        optimizer.zero_grad()
        output = model(batch_X)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader):.4f}")

Starting training...
Epoch 1, Loss: 0.2193
Epoch 2, Loss: 0.0231
Epoch 3, Loss: 0.0142
Epoch 4, Loss: 0.0053
Epoch 5, Loss: 0.0104
Epoch 6, Loss: 0.0026
Epoch 7, Loss: 0.0023
Epoch 8, Loss: 0.0015
Epoch 9, Loss: 0.0034
Epoch 10, Loss: 0.0016


In [8]:
# Save the model
#torch.save(model.state_dict(), "../models/motor_model_scratch3.pth") # uncommet this line to update the main model
#print("Model saved as motor_model_scratch3.pth")

Model saved as motor_model_scratch3.pth


In [15]:
import serial
import numpy as np
import time

# Update with your Pico 2 port (e.g., /dev/ttyACM0)
ser = serial.Serial('/dev/ttyACM0', 115200, timeout=1)

def get_live_window():
    window = []
    print("Collecting 128 samples...")
    while len(window) < 128:
        line = ser.readline().decode('utf-8').strip()
        try:
            # Assuming line is "raw,baseline,delta,current"
            parts = line.split(',')
            if len(parts) >= 4:
                val = float(parts[3]) # This is your 'current'
                window.append(val)
        except:
            continue
    return np.array(window)

live_window = get_live_window()

In [16]:
# inference.py
import torch
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# 1. Load the model
model = MotorFaultCNN() # Ensure this class is defined exactly as in your training script
model.load_state_dict(torch.load("../models/motor_model_scratch3.pth"))
model.eval()

# 2. Simulate or capture live data
# This is a dummy example; replace with your actual sensor reading loop
# 2. Simulate or capture live data
def classify_live_data(live_window):
    # CRITICAL: Scale the live window to [0, 1]
    # You must use the same min/max values used during training
    # For example, if your training data spanned 0 to 200:
    data_normalized = (live_window - 0.0) / 200.0 
    
    data = np.array(data_normalized).reshape(1, 1, 128)
    data = torch.tensor(data, dtype=torch.float32)
    
    with torch.no_grad():
        output = model(data)
        prediction = torch.argmax(output, dim=1).item()
        
    return "Normal" if prediction == 0 else "Slowed"
# Test it!
print(f"Prediction: {classify_live_data(live_window)}")

Prediction: Normal


In [11]:
# Instantiate and load your trained model
model = MotorFaultCNN()
model.load_state_dict(torch.load("../models/motor_model_scratch3.pth"))
model.eval()

# Dummy input matching your shape [batch_size=1, channels=1, length=128]
dummy_input = torch.randn(1, 1, 128)

# Export to ONNX
onnx_path = "../models/motor_model.onnx"
torch.onnx.export(
    model, 
    dummy_input, 
    onnx_path, 
    export_params=True,
    opset_version=11,
    input_names=['input'], 
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)
print(f"Model successfully exported to {onnx_path}")

/tmp/ipykernel_14452/4007898399.py:11: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0709 22:40:37.178000 14452 site-packages/torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `MotorFaultCNN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MotorFaultCNN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/home/jonny/miniconda3/envs/qai_hub/lib/python3.10/copyreg.py:101: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 11).
Failed to convert the model to the target version 11 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/home/jonny/miniconda3/envs/qai_hub/lib/python3.10/site-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
  File "/home/jonny/miniconda3/envs/qai_hub/lib/python3.10/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/home/jonny/miniconda3/envs/qai_hub/lib/python3.10/site-packages/onnxscript/version_conver

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Model successfully exported to ../models/motor_model.onnx


In [12]:
from onnx_tf.backend import prepare
import tensorflow as tf
import onnx
import numpy as np

# Load ONNX and convert to TF
onnx_model = onnx.load("../models/motor_model.onnx")
tf_rep = prepare(onnx_model)
tf_path = "../models/motor_model_tf"
tf_rep.export_graph(tf_path)

# Convert to TFLite with integer quantization
converter = tf.lite.TFLiteConverter.from_saved_model(tf_path)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Provide a representative dataset generator using your training/calibration data
def representative_dataset_gen():
    # Use a small batch of your actual normalized training windows (X_n / X_s)
    # X is your tensor of shape [N, 1, 128] from your loader setup
    for i in range(min(100, len(X))):
        sample = X[i].numpy().reshape(1, 1, 128)
        # TFLite expects float32 arrays for representative calibration
        yield [sample.astype(np.float32)]

converter.representative_dataset = representative_dataset_gen
# Ensure full integer quantization for microcontroller hardware accelerators
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.float32 # Keep input float if your MicroPython script normalizes 0-1, or set to int8
converter.inference_output_type = tf.int8

tflite_quantized_model = converter.convert()

with open("../models/motor_model_quantized.tflite", "wb") as f:
    f.write(tflite_quantized_model)

print("Quantized TFLite model saved successfully!")

I0000 00:00:1783617039.564302   14452 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1783617039.601104   14452 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


ImportError: cannot import name 'mapping' from 'onnx' (/home/jonny/miniconda3/envs/qai_hub/lib/python3.10/site-packages/onnx/__init__.py)